# NIT-4 · 3 — Four experiments, as they actually happened

**The failure this notebook prevents.** It is 1982. Somebody at the Department of
Health and Human Services has four final reports on the desk — New Jersey, Rural, Gary,
Seattle-Denver — and a memo asking for *the* number. The reports were written by four
contractors over fourteen years. One of them measured against a control group whose
circumstances changed a year in. One of them changed what it meant by "hours worked"
half-way through. One of them enrolled a cohort it later cancelled. All of them lost a
quarter of their families before the end, and not the same quarter in every arm.

None of that is in the four numbers. It is in the reports, if you read all of them,
and the person with the memo has an afternoon.

This notebook runs the four experiments from one synthetic world and files each one
as it goes: the plan, committed; the departures from it, recorded with their reason
when they happen rather than remembered afterwards; the delivery and attrition checks
that decide whether the arms are still comparable; and the readout, attached to the
estimand it actually answers. At the end there is a table with one line per experiment
that says whether its readout is the analysis its plan committed to — and that table,
not the four numbers, is what notebook 4 pools.

| | reaches into |
|---|---|
| a scope per party, and the thing that files into it | `io.Catalog`, `io.Program`, `build.ExperimentBuilder` |
| the plan, frozen | `io.ExperimentRun`, `design.ArmAssignment`, `design.Occupancy` |
| what a party means by a term | `io.DefinitionRegistry`, `io.Change` |
| did the units arrive in their arms | `diagnose.check_delivery`, `diagnose.attrition`, `identify.lee_bounds` |
| a dose is not a switch | `identify.compliance`, `identify.response_to_dose` |
| the neighbour or the effect | `design.factorial` |
| the readout, and whether it is the plan's | `identify.intention_to_treat`, `calibrate.Measurement`, `ExperimentRun.conformance`, `build.readouts_across` |

In [ ]:
import sys

sys.path.insert(0, ".")

import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

import guarantee as W
from axiom.build import ExperimentBuilder, readouts_across
from axiom.calibrate import Measurement
from axiom.core import Intervention, TimeWindow
from axiom.design import Occupancy, factorial
from axiom.diagnose import Attrition, DeliveryReport, attrition, check_delivery
from axiom.identify import LeeBounds, compliance, intention_to_treat, lee_bounds, response_to_dose
from axiom.io import Catalog, Change, ExperimentRun, Program
from axiom.display import enable, show, table

from _style import caption, compare, density, intervals, mark_x, points

enable();  # every axiom result renders itself from here on

programme = W.run_programme()
table([[W.site_title(s.name), s.party, s.experiment, s.started, f"{s.n_families:,}", len(s.plans),
        ", ".join(f"{d}y" for d, _ in s.durations)]
       for s in programme.sites],
      headers=("site", "party", "experiment", "began", "families", "plans", "enrolment offered"))

## 1 · Four parties, one catalog

Each contractor is a party and each party gets a scope. An artifact belongs to
exactly one scope, and the only way it reaches another is a transfer with a ledger
line — which is the rule that stops Gary's numbers from being quietly averaged with
New Jersey's on the strength of both being in the same directory.

`ExperimentBuilder` is a facade over one party's scope. It adds no statistics; it
puts the typed objects each layer produces under one experiment id and keeps the
run's conformance verdict standing over them.

In [ ]:
catalog = Catalog(Path(tempfile.mkdtemp()) / "income-maintenance")
builders: dict[str, ExperimentBuilder] = {}
for site in programme.sites:
    party = Program(party=site.party, program="income-maintenance", started=str(site.started),
                    description=f"{site.experiment}: {site.city}")
    catalog.register(party)
    builders[site.name] = ExperimentBuilder(catalog, party)
table([[p.scope, p.started, p.description] for p in catalog.programs()],
      headers=("scope", "began", "what it is"), title="the catalog")

## 2 · Planning and committing

Every site plans the same shape of experiment: a randomization rule (the
`ArmAssignment` from notebook 2, hashed roster and all), an occupancy (which cities,
which years, which lever), the enrolment window, and the estimand it commits to. The
estimand is the contrast at the site's **mean plan** — the average guarantee and tax
rate over its plan arms — because what a site's intention-to-treat contrast estimates is
the average of its plans against nothing, and on a linear surface that is the contrast
at the mean. Saying so in the estimand is what lets notebook 4 check it.

`commit` freezes a hash over the roles and the estimand. From then on any change to a
planned role has to go through `deviate`, with a reason.

In [ ]:
calendar = {
    "new_jersey": (("trenton", "paterson", "jersey_city", "scranton"), TimeWindow(start=0, stop=4)),
    "rural": (("iowa", "north_carolina"), TimeWindow(start=2, stop=5)),
    "gary": (("gary",), TimeWindow(start=3, stop=7)),
    "seattle_denver": (("seattle", "denver"), TimeWindow(start=3, stop=14)),
}


def mean_plan(run: W.SiteRun, years: int = 3) -> tuple[float, float]:
    treated = run.frame[(run.frame.assigned > 0) & (run.frame.years == years)]
    return float(treated.guarantee.mean()), float(treated.tax_rate.mean())


runs: dict[str, ExperimentRun] = {}
estimands = {}
for site in programme.sites:
    run = programme[site.name]
    units, window = calendar[site.name]
    cohorts = [(f"{site.experiment}", 3)] if len(site.durations) == 1 else [
        (f"{site.experiment}-3Y", 3), (f"{site.experiment}-5Y", 5), (f"{site.experiment}-20Y", 20)]
    for experiment, years in cohorts:
        plan = mean_plan(run, years)
        estimand = W.national_estimand(plan=plan, years=years, name=f"{experiment.lower().replace('-', '_')}_mean_plan")
        occupancy = Occupancy(experiment=experiment, units=units, window=window, treatments=("plan",))
        planned = builders[site.name].plan(
            experiment, estimand=estimand,
            roles={"assignment": run.assigned.spec, "occupancy": occupancy,
                   "window": W.enrolment_window(years), "reference": estimand.reference,
                   "outcome_definition": W.hours_definition("self_report")},
            at=f"{site.started}-03-01T00:00:00+00:00",
        )
        committed = builders[site.name].commit(planned, at=f"{site.started}-06-01T00:00:00+00:00",
                                               note="design review signed off")
        runs[experiment] = builders[site.name].start(committed, at=f"{site.started}-09-01T00:00:00+00:00")
        estimands[experiment] = estimand
table([[k, r.scope, r.stage, f"G = {estimands[k].intervention.doses['guarantee']:.2f}, t = {estimands[k].intervention.doses['tax_rate']:.2f}",
        estimands[k].window.stop, r.plan_hash[:12] + "…"] for k, r in runs.items()],
      headers=("experiment", "scope", "stage", "committed estimand: contrast at", "years", "plan hash"))

## 3 · What each party means by "hours worked"

Every site registers what it means by the outcome. Three of them mean the family
head's own report at the quarterly interview. Gary, after an audit in 1973 found
earnings under-reported, moved to hours reconciled against employer records — for the
rest of the experiment, and for none of the data before it. Registering the change is
a diff with a date; nobody has to remember it, and notebook 4 will find it.

In [ ]:
for site in programme.sites:
    builders[site.name].define("hours_worked", W.hours_definition("self_report"), at=f"{site.started}-03-01")
gary = builders["gary"]
gary.define("hours_worked", W.hours_definition("employer_verified"), at="1973-06-01",
            note="reconciled against employer and unemployment-insurance records after the earnings audit")
table([[d.version, d.registered, d.digest[:12] + "…", d.note or "—"]
       for d in gary.definitions.history(gary.program, "hours_worked")],
      headers=("version", "registered", "digest", "note"), title="indiana/income-maintenance: hours_worked")
for change in gary.definitions.changes(gary.program, "hours_worked"):
    assert isinstance(change, Change)
    print(f"{change.at}: v{change.from_version} -> v{change.to_version}")
    for field, moved in change.changed.items():
        print(f"   {field}: {moved}")
runs["GARY-1971"] = runs["GARY-1971"].deviate(
    "outcome_definition", W.hours_definition("employer_verified"),
    reason="hours reconciled against employer records from June 1973; earlier quarters are self-reported",
    at="1973-06-01T00:00:00+00:00",
)
print("\nGary's run now carries", len(runs["GARY-1971"].deviations), "deviation, state",
      runs["GARY-1971"].deviations[0].state)

## 4 · The departures, filed when they happened

**New Jersey, January 1969.** The state adopts AFDC-UP, extending welfare to families
with an unemployed father. Control families now have a benefit they did not have when
the plan was committed. The reference of the estimand — *no plan* — has moved to *no
plan, with AFDC-UP available*. That is a deviation on the `reference` role, and its
state is `violated` rather than `unverified`: nobody thinks the contrast is unchanged,
because the whole point of a welfare alternative is that it changes how much control
families work.

**Seattle-Denver, 1980.** The twenty-year cohort — six per cent of treated families,
enrolled to see whether a permanent guarantee behaves like a temporary one — is
cancelled when Congress declines to fund the remaining years. That run is
**abandoned**, with the reason, and never reaches a readout.

In [ ]:
afdc_up = Intervention(doses={"guarantee": 0.0, "tax_rate": 0.0}, version="afdc_up_available")
runs["NJ-1968"] = runs["NJ-1968"].deviate(
    "reference", afdc_up,
    reason="New Jersey adopted AFDC-UP in January 1969; control families gained a welfare alternative",
    state="violated", at="1969-01-15T00:00:00+00:00",
)
runs["SIME-DIME-1971-20Y"] = runs["SIME-DIME-1971-20Y"].abandon(
    "the twenty-year cohort was cancelled in 1980 when the remaining payments were not funded",
    at="1980-06-30T00:00:00+00:00",
)
builders["seattle_denver"].file(runs["SIME-DIME-1971-20Y"])
d = runs["NJ-1968"].deviations[0]
table([["role", d.role], ["planned", d.planned[:16] + "…"], ["realized", d.realized[:16] + "…"],
       ["reason", d.reason], ["state", d.state], ["filed", d.at[:10]]],
      headers=("", "New Jersey's departure, as recorded"))
print("\n" + str(runs["SIME-DIME-1971-20Y"]), "—", runs["SIME-DIME-1971-20Y"].history[-1].note)

## 5 · Did the families arrive in their arms

The first check is not about the model. A split that is not the split the rule asked
for, or arms that differ on what was recorded before enrolment, invalidates everything
downstream. `check_delivery` runs the sample-ratio test and the balance test against
the `Assigned` from the rule, at the alpha a check that runs on every experiment can
afford.

In [ ]:
rows = []
for site in programme.sites:
    run = programme[site.name]
    covariates = {c: run.families[c].to_numpy() for c in ("pre_hours", "earnings", "schooling")}
    report: DeliveryReport = check_delivery(run.assigned, covariates=covariates)
    rows.append([W.site_title(site.name), run.assigned.spec.allocation.n_arms, f"{report.ratio.p_value:.3f}",
                 f"{report.covariates.worst_smd:.3f}", ", ".join(report.covariates.imbalanced) or "none",
                 report.verdict().status])
table(rows, headers=("site", "arms", "sample-ratio p", "worst smd", "imbalanced", "verdict"),
      title="delivery: were the families randomized as the rule says")
print("a passing check leaves 'random_assignment' unverified, not satisfied:",
      report.ratio.assumption().state)

## 6 · Reached is not observed

Every site lost families before the end, and not evenly: control families and
families on thin plans had less reason to keep answering interviews. That is not a
compliance problem — the plan reached everybody it was assigned to — it is a
**selection** problem, and it has to be answered with a bound rather than a number,
because the families who left are not a random draw of the families who stayed.

In [ ]:
rows, checks = [], {}
for site in programme.sites:
    f = programme[site.name].frame
    assigned = {"control": int((f.assigned == 0).sum()), "plan": int((f.assigned == 1).sum())}
    observed = {"control": int(f.loc[f.assigned == 0, "observed"].sum()), "plan": int(f.loc[f.assigned == 1, "observed"].sum())}
    check: Attrition = attrition(assigned, observed)
    checks[site.name] = check
    rows.append([W.site_title(site.name), f"{check.overall:.1%}",
                 f"{observed['control'] / assigned['control']:.1%}", f"{observed['plan'] / assigned['plan']:.1%}",
                 f"{check.differential:.3f}", f"{check.p_value:.2g}", check.verdict().status])
table(rows, headers=("site", "observed overall", "control observed", "plan observed", "gap", "p", "verdict"),
      title="attrition: did the outcome reach the analyst")
show(checks["seattle_denver"].ledger_line())

`blocked` at three of the four sites — the Rural gap of eight points is `p = 0.012`,
under the five per cent everybody reaches for and above the one-in-a-thousand a check
that runs on every experiment can afford — and never `downgraded`: no assumption over observables
licenses a contrast between two arms whose survivors are different populations. What
licenses a number is a **bound** — the identified set for the effect on the families who
would have reported either way, which `lee_bounds` computes by trimming the arm that
kept more of its families. The width of the bound is the price of the imbalance, and it
is not small.

In [ ]:
rows, forest = [], []
for site in programme.sites:
    f = programme[site.name].frame
    bounds = lee_bounds(f, y="hours_observed", assignment="assigned", selection="observed")
    assert isinstance(bounds, LeeBounds)
    truth = intention_to_treat(f, "hours_change", "assigned").estimate
    rows.append([W.site_title(site.name), f"{bounds.naive:.0f}", f"[{bounds.lower:.0f}, {bounds.upper:.0f}]",
                 f"{bounds.width:.0f}", f"{bounds.trimmed:.1%} of {bounds.trimmed_arm}", f"{truth:.0f}"])
    forest.append((W.site_title(site.name), bounds.naive, bounds.lower, bounds.upper))
table(rows, headers=("site", "naive contrast", "Lee bounds", "width", "trimmed", "the truth (synthetic world)"),
      title=f"hours change, plan against control, among families still answering ({W.OUTCOME_UNIT})")
fig = intervals(
    forest, ref=0.0, ref_label="no effect",
    title="What the missing families could have changed",
    subtitle="the naive contrast among survivors, inside the Lee bounds on the always-responder effect",
    x_title=f"hours change, plan against control ({W.OUTCOME_UNIT})",
)
caption(fig, "The bars are not confidence intervals. They are the set of answers consistent with "
             "the families who left, under one assumption — that a family the plan kept in the "
             "study would also have stayed under no plan. Every bound still excludes zero, which "
             "is the finding that survives; the point inside it is not.")

## 7 · A dose is not a switch

Gary's families reported less than they earned, so the tax rate the experiment
actually applied to their earnings was lower than the one they were assigned. Ask the
two questions the library has for a partly-delivered treatment and watch them disagree.

`compliance` sees a switch: was the family enrolled or not. Everybody assigned a plan
was enrolled, so it reports perfect compliance and hands the intention-to-treat effect
back as *the* effect. `response_to_dose` sees an amount: the tax rate delivered against
the one assigned, and reports the effect **per unit of tax rate** over the families the
assignment moved — a different estimand, on a different scale, and not the one the
plan was committed to.

In [ ]:
g = programme["gary"].observed.assign(enrolled=lambda d: (d.assigned > 0).astype(float))
as_switch = compliance(g, "hours_observed", "assigned", "enrolled")
print("as a switch:", as_switch.verdict().status, "—", as_switch.unverified.reason[:110] if as_switch.unverified else "")
as_dose = response_to_dose(g, "hours_observed", "assigned", "delivered_tax")
print(as_dose.summary())
print("\nverdict:", as_dose.verdict().status, "| assumptions:", [a.name for a in as_dose.assumptions()])
assigned_rate = float(g.loc[g.assigned > 0, "tax_rate"].mean())
print(f"\nmean tax rate assigned to Gary's plan families: {assigned_rate:.3f}; delivered: {as_dose.shift:.3f} "
      f"({as_dose.shift / assigned_rate:.0%} of it)")
print(f"itt = derivative x shift: {as_dose.derivative.estimate * as_dose.shift:.1f} against {as_dose.itt.estimate:.1f}")

In [ ]:
delivered = g.loc[g.assigned > 0, "delivered_tax"]
fig = density(
    {"tax rate delivered to Gary's plan families": delivered.to_numpy()},
    title="The dose Gary's families received",
    subtitle="the rate actually applied to true earnings, against the rates the plans assigned",
    x_title="tax rate on true earnings",
)
for t in sorted(g.loc[g.assigned > 0, "tax_rate"].unique()):
    mark_x(fig, float(t), text=f"assigned {t:.1f}", color=W.DECISION_COLOR)
caption(fig, "Nobody received the rate they were assigned. The intention-to-treat contrast is "
             "still the effect of *offering* the plan — which is what a government can do — but "
             "reading it as the effect of a 50 % tax rate would be wrong by a third, and only the "
             "dose-based report says so.")

## 8 · The neighbour or the effect

Notebook 2 found one overlap on the calendar: the training subsidy randomized inside
Seattle-Denver. The collision was `concurrent` — a cost, not a conflict — *on the
assumption that the two treatments do not interact*. That assumption's `challenged_by`
names a factorial analysis. Run it.

In [ ]:
s = programme["seattle_denver"].observed
both = factorial(s, "hours_observed", "assigned", "training")
print(both.summary())
print("\nverdict:", both.verdict().status)
print(" ", both.verdict().reason)

`unverified`, never `identified`: an interaction is about twice as hard to see as a
main effect of the same size, and a study powered for the plan's effect cannot rule
out an interaction a seventh of its size. The check did not find one, and it says what
it could have found. The collision stays a cost.

## 9 · Reading the experiments

Each experiment's readout is the intention-to-treat contrast among families still
answering — the number the four final reports actually published — as a
`Measurement` against the estimand the plan committed to. New Jersey's is the
exception: its analysts, knowing the reference moved, declare the estimand their
readout answers, *no plan with AFDC-UP available*, and file against that. The run
records the mismatch rather than refusing it.

In [ ]:
measurements = {}
cohort_of = {"NJ-1968": ("new_jersey", 3), "RURAL-1970": ("rural", 3), "GARY-1971": ("gary", 3),
             "SIME-DIME-1971-3Y": ("seattle_denver", 3), "SIME-DIME-1971-5Y": ("seattle_denver", 5)}
for experiment, (site_name, years) in cohort_of.items():
    frame = programme[site_name].cohort(years)
    itt = intention_to_treat(frame, "hours_observed", "assigned")
    answers = estimands[experiment]
    if experiment == "NJ-1968":
        answers = answers.model_copy(update={"name": "nj_1968_mean_plan_vs_afdc_up", "reference": afdc_up})
    measurements[experiment] = Measurement(
        estimand=answers, estimate=itt.estimate, se=itt.se, definition="wald", mass=0.95,
        method="intention_to_treat", n_units=itt.n, source=experiment,
    )
    runs[experiment] = builders[site_name].read(runs[experiment], measurements[experiment],
                                                at=f"{W.SITE_BY_NAME[site_name].started + years}-12-01T00:00:00+00:00")
table([[k, f"{m.estimate:.0f} ± {m.se:.0f}", m.interval.text(), m.n_units, m.estimand.name]
       for k, m in measurements.items()],
      headers=("experiment", "hours change", "95% interval", "families", "answers"))

## 10 · Is the readout the analysis the plan committed to?

One line per run. `conformance()` is a `Verdict` in the vocabulary identification
already uses, because whether two readouts may be compared is a question about what
licenses the comparison. `readouts_across` collects the runs a programme may decide
on — and skips the ones the run itself calls `blocked`, because a readout that answers
a different estimand than the one committed is not evidence about anything.

In [ ]:
for experiment, run in runs.items():
    builders[cohort_of.get(experiment, ("seattle_denver", 20))[0]].file(run)
rows = []
for experiment, run in runs.items():
    verdict = run.conformance()
    rows.append([experiment, run.stage, len(run.deviations), verdict.status, (verdict.reason or "—")[:90]])
table(rows, headers=("experiment", "stage", "deviations", "conformance", "why"), title="the six runs")
licensed = readouts_across(list(builders.values()))
print("\nreadouts a programme may decide on:", [r.key for r in licensed])
print("filed in the catalog:", len(catalog), "artifacts across", len(catalog.scopes()), "scopes")

## What notebook 3 established

1. **Four parties, four scopes, one catalog.** Every plan, rule, occupancy, definition
   and readout is filed under the party that produced it, with a content hash, and
   the runs carry their own history.
2. **The departures are in the record, not in the reports.** New Jersey's reference
   moved and the deviation says so with its date and its state; Seattle-Denver's
   twenty-year cohort was abandoned with its reason; Gary's definition of hours
   changed in June 1973 and the change is a field-level diff.
3. **Randomization held, and the outcome did not come back evenly.** Delivery is
   `identified` at every site; attrition is `blocked` at three of four, with control
   families lost at a higher rate than plan families everywhere. The number that
   survives is a Lee bound, and every bound excludes zero.
4. **Gary's plan families received a tax rate two-thirds of the one assigned.** As a
   switch the experiment looks perfectly complied with; as a dose it is a
   derivative over a moved population, and the intention-to-treat contrast is the
   effect of offering the plan, not of the plan.
5. **The training programme did not interact with the plan** — as far as a study
   this size can tell, which is not very far, and the verdict says `unverified`
   rather than pretending otherwise.
6. **Five readouts, four licensed to be compared as planned.** Three are
   `identified`, one `downgraded` — Gary's, carrying its changed definition as an
   `unverified` assumption — one `blocked` because it answers a reference nobody
   committed to, and one never read. Notebook 4 starts from that table.